In [1]:
from dp_model.model_files.sfcn import SFCN
from dp_model import dp_loss as dpl
from dp_model import dp_utils as dpu
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Example
model = SFCN(output_dim=2, channel_number=[28, 58, 128, 256, 256, 64])
model = torch.nn.DataParallel(model)
fp_ = 'run_20191008_00_epoch_last.p'
model.load_state_dict(torch.load(fp_))
model.cuda()

# Example data: some random brain in the MNI152 1mm std space
data = np.random.rand(182, 218, 182)
y = torch.tensor([1]) # Assuming Sex is Male (0=Female, 1=Male)

# Preprocessing
data = data/data.mean()
data = dpu.crop_center(data, (160, 192, 160))

# Move the data from numpy to torch tensor on GPU
sp = (1,1)+data.shape
data = data.reshape(sp)
input_data = torch.tensor(data, dtype=torch.float32).cuda()
print(f'Input data shape: {input_data.shape}')
print(f'dtype: {input_data.dtype}')

# Evaluation
model.eval() # Don't forget this. BatchNorm will be affected if not in eval mode.
with torch.no_grad():
    output = model(input_data)

# Output, loss, visualisation
x = output[0].cpu().reshape([1, -1])
loss = F.nll_loss(x, y)

# Prediction, Visualisation and Summary
x = np.exp(x.numpy().reshape(-1))

print('\nPredicted probability: \nFemale\t%.2f%%,\nMale\t%.2f%%'%(x[0]*100, x[1]*100))

Input data shape: torch.Size([1, 1, 160, 192, 160])
dtype: torch.float32

Predicted probability: 
Female	34.73%,
Male	65.27%


In [3]:
import os
import re
import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import copy
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.decomposition import PCA
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset, TensorDataset
from scipy.ndimage import zoom
import shutil

# --- Model Imports ---
from dp_model.model_files.sfcn import SFCN
from networks import SpectralViT 

# --- Configuration ---
MNI_DIR = '/data/Ali/SpectralViT/data/IXI_extracted/'
CSV_PATH = '/data/Ali/SpectralViT/data/IXI_extracted/IXI.csv'
CACHE_DIR = '/data/Ali/SpectralViT/data/IXI_cached_npy/'
CROP_SIZE = (160, 192, 160)
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")

# Hyperparams
SFCN_LR = 1e-5
SPEC_LR = 1e-4
EPOCHS_SFCN = 20 # Increased to give early stopping room
EPOCHS_SPEC = 100 # Increased to give early stopping room
PATIENCE = 5
BATCH_SIZE = 2
N_COMP = 64 
EVAL_SNRS = [None, 9.0, 4.0, 1.0, 0.25, 0.111] 

# --- 1. Data Processing & Noise ---

def pre_process_data(paths, cache_dir):
    if not os.path.exists(cache_dir): os.makedirs(cache_dir)
    print("Checking/Pre-processing images to .npy...")
    for p in tqdm(paths):
        fname = os.path.basename(p).replace('.nii.gz', '.npy')
        cp = os.path.join(cache_dir, fname)
        if not os.path.exists(cp):
            img = nib.load(p)
            voxel_size = np.array(img.header.get_zooms()[:3])
            data = img.get_fdata()
            zoom_factors = voxel_size / np.array([1.0, 1.0, 1.0])
            resampled = zoom(data, zoom_factors, order=1)
            out = np.zeros(CROP_SIZE, dtype=np.float32)
            slices_in, slices_out = [], []
            for rs, ts in zip(resampled.shape, CROP_SIZE):
                if rs >= ts:
                    start = (rs - ts) // 2
                    slices_in.append(slice(start, start + ts))
                    slices_out.append(slice(0, ts))
                else:
                    pad = (ts - rs) // 2
                    slices_in.append(slice(0, rs))
                    slices_out.append(slice(pad, pad + rs))
            out[tuple(slices_out)] = resampled[tuple(slices_in)]
            std = np.std(out)
            norm = (out - np.mean(out)) / (std + 1e-8)
            np.save(cp, norm.astype(np.float32))

def apply_rician_noise(x, snr):
    if snr is None: return x
    x_min = x.min()
    x_pos = x - x_min
    signal_mean = np.mean(x_pos[x_pos > 0])
    sigma = signal_mean / snr
    n1 = np.random.normal(0, sigma, x.shape)
    n2 = np.random.normal(0, sigma, x.shape)
    x_noisy = np.sqrt((x_pos + n1)**2 + n2**2)
    return ((x_noisy - np.mean(x_noisy)) / (np.std(x_noisy) + 1e-8)).astype(np.float32)

def get_data_lists(data_dir, csv_path):
    df = pd.read_csv(csv_path)
    id_col = [c for c in df.columns if 'ID' in c.upper()][0]
    sex_col = [c for c in df.columns if 'SEX' in c.upper()][0]
    sex_lookup = dict(zip(df[id_col].astype(int), df[sex_col].map({1: 1, 2: 0})))
    paths, labels = [], []
    all_files = sorted([f for f in os.listdir(data_dir) if f.endswith('.nii.gz')])
    for f in all_files:
        match = re.search(r'(\d+)', f)
        if match and int(match.group(1)) in sex_lookup:
            paths.append(os.path.join(CACHE_DIR, f.replace('.nii.gz', '.npy')))
            labels.append(sex_lookup[int(match.group(1))])
    return np.array(paths), np.array(labels)

class IXIDataset(Dataset):
    def __init__(self, paths, labels, snr=None):
        self.paths = paths
        self.labels = labels
        self.snr = snr
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        x = np.load(self.paths[idx])
        x = apply_rician_noise(x, self.snr)
        return torch.from_numpy(x).unsqueeze(0), torch.tensor(self.labels[idx]).long()

# --- 2. Training Function for SpectralViT ---

def train_spec_vit(tr_pca, tr_y, vl_pca, vl_y, n_comp):
    model = SpectralViT(n_inputs=n_comp, embed_dim=16, use_mode_weights=True).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=SPEC_LR)
    criterion = nn.BCEWithLogitsLoss()
    
    train_loader = DataLoader(TensorDataset(torch.tensor(tr_pca).float(), torch.tensor(tr_y).float()), batch_size=8, shuffle=True)
    val_pca_t = torch.tensor(vl_pca).float().to(device)
    val_y_t = torch.tensor(vl_y).float().to(device)

    best_loss = float('inf')
    best_model_wts = None
    counter = 0

    for epoch in range(EPOCHS_SPEC):
        model.train()
        for b_x, b_y in train_loader:
            b_x, b_y = b_x.to(device), b_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(b_x), b_y)
            loss.backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(val_pca_t), val_y_t).item()
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1
            if counter >= PATIENCE:
                break
                
    model.load_state_dict(best_model_wts)
    return model

# --- 3. Main Execution Loop ---

raw_nii_files = [os.path.join(MNI_DIR, f) for f in os.listdir(MNI_DIR) if f.endswith('.nii.gz')]
pre_process_data(raw_nii_files, CACHE_DIR)

all_npy_paths, all_labels = get_data_lists(MNI_DIR, CSV_PATH)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
results = []

for fold, (train_idx, test_idx) in enumerate(kf.split(all_npy_paths, all_labels), 1):
    print(f"\n=== FOLD {fold} ===")
    
    # Split train_idx into sub_train and validation for early stopping
    tr_idx, val_idx = train_test_split(train_idx, test_size=0.1, stratify=all_labels[train_idx], random_state=0)

    # --- SFCN Setup ---
    sfcn = SFCN(output_dim=2, channel_number=[28, 58, 128, 256, 256, 64])
    state_dict = torch.load('./run_20191008_00_epoch_last.p', map_location=device)
    sfcn.load_state_dict({k.replace('module.', ''): v for k, v in state_dict.items()})
    sfcn.to(device)
    
    optimizer_sfcn = optim.Adam(sfcn.parameters(), lr=SFCN_LR)
    train_ds = IXIDataset(all_npy_paths[tr_idx], all_labels[tr_idx], snr=None)
    val_ds   = IXIDataset(all_npy_paths[val_idx], all_labels[val_idx], snr=None)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

    print("Fine-tuning SFCN with Early Stopping...")
    best_sfcn_loss = float('inf')
    best_sfcn_wts = None
    sfcn_counter = 0

    for epoch in range(EPOCHS_SFCN):
        sfcn.train()
        for b_x, b_y in train_loader:
            optimizer_sfcn.zero_grad()
            out = sfcn(b_x.to(device))
            if isinstance(out, list): out = out[0]
            loss = F.nll_loss(out.view(out.size(0), -1), b_y.to(device))
            loss.backward()
            optimizer_sfcn.step()
        
        sfcn.eval()
        val_loss = 0
        with torch.no_grad():
            for b_x, b_y in val_loader:
                out = sfcn(b_x.to(device))
                if isinstance(out, list): out = out[0]
                val_loss += F.nll_loss(out.view(out.size(0), -1), b_y.to(device)).item()
        val_loss /= len(val_loader)

        if val_loss < best_sfcn_loss:
            best_sfcn_loss = val_loss
            best_sfcn_wts = copy.deepcopy(sfcn.state_dict())
            sfcn_counter = 0
        else:
            sfcn_counter += 1
            if sfcn_counter >= PATIENCE: break
    
    sfcn.load_state_dict(best_sfcn_wts)

    # --- SpectralViT Setup ---
    print("Training SpectralViT with Early Stopping...")
    X_tr_flat = np.array([np.load(p).flatten() for p in all_npy_paths[tr_idx]])
    X_vl_flat = np.array([np.load(p).flatten() for p in all_npy_paths[val_idx]])
    
    pca = PCA(n_components=N_COMP, whiten=True).fit(X_tr_flat)
    tr_pca = pca.transform(X_tr_flat)
    vl_pca = pca.transform(X_vl_flat)
    
    spec_vit = train_spec_vit(tr_pca, all_labels[tr_idx], vl_pca, all_labels[val_idx], N_COMP)

    # --- Evaluation ---
    for snr in EVAL_SNRS:
        snr_label = "Real" if snr is None else f"SNR {snr:.3f}"
        y_true = all_labels[test_idx]
        sfcn.eval(); spec_vit.eval()
        sfcn_probs, spec_probs = [], []
        
        for i in range(len(test_idx)):
            img = np.load(all_npy_paths[test_idx[i]])
            img_noisy = apply_rician_noise(img, snr)
            
            with torch.no_grad():
                # SFCN
                vol_t = torch.from_numpy(img_noisy).float().unsqueeze(0).unsqueeze(0).to(device)
                out_sfcn = sfcn(vol_t)
                if isinstance(out_sfcn, list): out_sfcn = out_sfcn[0]
                sfcn_probs.append(torch.exp(out_sfcn)[:, 1].item())
                
                # SpecViT
                feat_pca = pca.transform(img_noisy.flatten().reshape(1, -1))
                feat_t = torch.tensor(feat_pca).float().to(device)
                out_spec = torch.sigmoid(spec_vit(feat_t))
                spec_probs.append(out_spec.item())

        auc_sfcn = roc_auc_score(y_true, sfcn_probs)
        auc_spec = roc_auc_score(y_true, spec_probs)
        print(f" [{snr_label}] SFCN AUC: {auc_sfcn:.4f} | SpectralViT AUC: {auc_spec:.4f}")
        results.append({'Fold': fold, 'Shift': snr_label, 'Model': 'SFCN', 'AUC': auc_sfcn})
        results.append({'Fold': fold, 'Shift': snr_label, 'Model': 'SpectralViT', 'AUC': auc_spec})

# --- Final Summary ---
df_res = pd.DataFrame(results)
summary = df_res.groupby(['Shift', 'Model'])['AUC'].mean().unstack()
print("\nFinal Robustness Comparison (Mean AUC):")
print(summary)

Checking/Pre-processing images to .npy...


100%|██████████| 581/581 [00:00<00:00, 107760.26it/s]


=== FOLD 1 ===


Fine-tuning SFCN with Early Stopping...
Training SpectralViT with Early Stopping...
 [Real] SFCN AUC: 0.9969 | SpectralViT AUC: 0.9213
 [SNR 9.000] SFCN AUC: 0.9978 | SpectralViT AUC: 0.9203
 [SNR 4.000] SFCN AUC: 0.9116 | SpectralViT AUC: 0.9150
 [SNR 1.000] SFCN AUC: 0.5419 | SpectralViT AUC: 0.8768
 [SNR 0.250] SFCN AUC: 0.4977 | SpectralViT AUC: 0.9163
 [SNR 0.111] SFCN AUC: 0.6222 | SpectralViT AUC: 0.8811

=== FOLD 2 ===
Fine-tuning SFCN with Early Stopping...
Training SpectralViT with Early Stopping...
 [Real] SFCN AUC: 0.9971 | SpectralViT AUC: 0.9194
 [SNR 9.000] SFCN AUC: 0.9959 | SpectralViT AUC: 0.9165
 [SNR 4.000] SFCN AUC: 0.9749 | SpectralViT AUC: 0.9149
 [SNR 1.000] SFCN AUC: 0.5013 | SpectralViT AUC: 0.8956
 [SNR 0.250] SFCN AUC: 0.4270 | SpectralViT AUC: 0.8556
 [SNR 0.111] SFCN AUC: 0.4263 | SpectralViT AUC: 0.8083

=== FOLD 3 ===
Fine-tuning SFCN with Early Stopping...
Training SpectralViT with Early Stopping...
 [Real] SFCN AUC: 0.9870 | SpectralViT AUC: 0.9358
 [S